# 03 WASDE Benchmark and Futures Signal Analysis

This notebook evaluates the satellite-implied soybean yield signals from `02_yield_prediction_models.ipynb` against WASDE benchmarks and soybean futures price responses.

The analysis has three parts:

1. **WASDE benchmark extraction**: extract August and November U.S. soybean projected yield, harvested area, production, and ending stocks from WASDE CSV files.
2. **Yield bracketing test**: test whether November WASDE yield falls within the July/August satellite-implied yield bracket.
3. **Futures signal test**: compare August satellite-implied yield with August WASDE yield, construct long/short/no-trade signals, and evaluate 3/5/10/20 trading-day and October-end futures returns.

In [ ]:
# ============================================================
# 0. Setup
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
import shutil

# Use the same mount root throughout this notebook.
# In this Colab session, the project was accessed through /content/ggdrive.
MOUNT_ROOT = Path("/content/ggdrive")
PROJECT_DIR = MOUNT_ROOT / "MyDrive" / "Soybean_Project_GEE"

WASDE_DIR = PROJECT_DIR / "wasde"
OUTPUT_TABLE_DIR = PROJECT_DIR / "outputs" / "tables"

WASDE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("WASDE_DIR exists:", WASDE_DIR.exists())
print("OUTPUT_TABLE_DIR exists:", OUTPUT_TABLE_DIR.exists())

## 1. Optional: copy uploaded WASDE CSV files into Google Drive

Files uploaded through the Colab file panel are stored in the temporary session directory. This cell copies any local WASDE CSV files into the project `wasde/` folder so they can be reused in future sessions.

In [ ]:
# ============================================================
# Optional: copy local uploaded WASDE CSV files into Drive
# ============================================================

local_wasde_files = sorted(Path(".").glob("oce-wasde-report-data-*.csv"))

print("Local uploaded WASDE files found:", len(local_wasde_files))
for f in local_wasde_files:
    dest = WASDE_DIR / f.name
    shutil.copy(f, dest)
    print("Copied:", f.name, "->", dest)

print("\nFiles currently in WASDE_DIR:")
for f in sorted(WASDE_DIR.glob("*.csv")):
    print(f.name)

## 2. Extract August and November WASDE soybean benchmarks

The function below filters the WASDE data to the current-crop-year U.S. soybean projected rows. August WASDE is used for the contemporaneous trading signal, while November WASDE is used as a later-season yield benchmark for the bracketing test.

In [ ]:
# ============================================================
# Function: extract WASDE projected soybean benchmark by month
# ============================================================

def extract_wasde_soybean_month(
    csv_files,
    forecast_month,
    start_year=2019,
    end_year=2025
):
    """
    Extract U.S. soybean projected WASDE rows for a given forecast month.

    forecast_month:
        8  = August WASDE
        11 = November WASDE
    """

    target_attrs = [
        "Area Harvested",
        "Yield per Harvested Acre",
        "Production",
        "Ending Stocks"
    ]

    month_name_map = {
        8: "aug",
        11: "nov"
    }
    month_name = month_name_map.get(forecast_month, f"m{forecast_month}")

    all_rows = []

    for path in csv_files:
        df = pd.read_csv(path)

        for col in [
            "ReportTitle",
            "Commodity",
            "Region",
            "MarketYear",
            "ProjEstFlag",
            "Attribute",
            "Unit"
        ]:
            if col in df.columns:
                df[col] = df[col].astype("string").str.strip()

        df["ForecastYear_num"] = pd.to_numeric(df["ForecastYear"], errors="coerce")
        df["ForecastMonth_num"] = pd.to_numeric(df["ForecastMonth"], errors="coerce")
        df["Value_numeric"] = pd.to_numeric(df["Value"], errors="coerce")

        # Crop year from MarketYear, e.g. 2021/22 -> 2021
        df["crop_year"] = (
            df["MarketYear"]
            .astype(str)
            .str.extract(r"^(\d{4})")[0]
            .astype(float)
        )

        subset = df[
            (df["ForecastMonth_num"] == forecast_month) &
            (df["ForecastYear_num"].between(start_year, end_year)) &
            (df["crop_year"] == df["ForecastYear_num"]) &
            (df["ProjEstFlag"] == "Proj.") &
            (df["ReportTitle"] == "U.S. Soybeans and Products Supply and Use (Domestic Measure)") &
            (df["Commodity"] == "Oilseed, Soybean") &
            (df["Region"] == "United States") &
            (df["Attribute"].isin(target_attrs))
        ].copy()

        subset["source_file"] = Path(path).name
        all_rows.append(subset)

    if not all_rows:
        raise ValueError("No WASDE files were provided.")

    raw = pd.concat(all_rows, ignore_index=True)

    if raw.empty:
        raise ValueError(
            f"No WASDE soybean rows found for forecast_month={forecast_month}. "
            "Check file paths, year range, and filters."
        )

    raw_display = (
        raw[
            [
                "source_file",
                "ReportDate",
                "ReleaseDate",
                "ReleaseTime",
                "ForecastYear",
                "ForecastMonth",
                "crop_year",
                "MarketYear",
                "ProjEstFlag",
                "Attribute",
                "Value",
                "Value_numeric",
                "Unit"
            ]
        ]
        .sort_values(["crop_year", "Attribute"])
        .reset_index(drop=True)
    )

    clean = (
        raw
        .sort_values(["crop_year", "Attribute", "source_file"])
        .drop_duplicates(subset=["crop_year", "Attribute"], keep="first")
        .copy()
    )

    wide = (
        clean
        .pivot_table(
            index=[
                "crop_year",
                "ReportDate",
                "ReleaseDate",
                "ReleaseTime",
                "ForecastYear",
                "ForecastMonth",
                "MarketYear",
                "ProjEstFlag"
            ],
            columns="Attribute",
            values="Value_numeric",
            aggfunc="first"
        )
        .reset_index()
    )

    wide.columns.name = None

    wide = wide.rename(columns={
        "crop_year": "year",
        "ReportDate": f"{month_name}_report_date",
        "ReleaseDate": f"{month_name}_release_date",
        "ReleaseTime": f"{month_name}_release_time",
        "ForecastYear": f"{month_name}_forecast_year",
        "ForecastMonth": f"{month_name}_forecast_month",
        "MarketYear": f"{month_name}_market_year",
        "ProjEstFlag": f"{month_name}_proj_est_flag",
        "Area Harvested": f"{month_name}_wasde_harvested_area",
        "Yield per Harvested Acre": f"{month_name}_wasde_yield",
        "Production": f"{month_name}_wasde_production",
        "Ending Stocks": f"{month_name}_wasde_ending_stocks"
    })

    wide["year"] = wide["year"].astype(int)
    wide[f"{month_name}_release_date"] = pd.to_datetime(
        wide[f"{month_name}_release_date"],
        errors="coerce"
    )

    wide = wide.sort_values("year").reset_index(drop=True)

    print(f"Filtered month {forecast_month} projected soybean rows:", len(raw))
    print(f"Wide benchmark rows:", len(wide))

    return raw_display, wide

In [ ]:
# ============================================================
# Read WASDE files and extract August / November benchmarks
# ============================================================

csv_files = sorted(WASDE_DIR.glob("oce-wasde-report-data-*.csv"))

print("WASDE CSV files found:", len(csv_files))
for p in csv_files:
    print(p.name)

aug_soy_raw, aug_wasde = extract_wasde_soybean_month(
    csv_files=csv_files,
    forecast_month=8,
    start_year=2019,
    end_year=2025
)

nov_soy_raw, nov_wasde = extract_wasde_soybean_month(
    csv_files=csv_files,
    forecast_month=11,
    start_year=2019,
    end_year=2025
)

display(aug_wasde.round(4))
display(nov_wasde.round(4))

## 3. Load satellite yield signal pivot

This table is produced by `02_yield_prediction_models.ipynb`. It contains the rolling-bias-adjusted July, August, and September satellite-implied yield signals.

In [ ]:
# ============================================================
# Load satellite signal pivot from notebook 02
# ============================================================

signal_file_candidates = [
    "signal_pivot_july_aug_sept.csv",
    "satellite_yield_signal_pivot_july_aug_sep.csv",
    "adjusted_satellite_yield_signal_pivot_july_aug_sep.csv"
]

signal_path = None
for name in signal_file_candidates:
    candidate = OUTPUT_TABLE_DIR / name
    if candidate.exists():
        signal_path = candidate
        break

if signal_path is None:
    print("Available CSV files in OUTPUT_TABLE_DIR:")
    for f in sorted(OUTPUT_TABLE_DIR.glob("*.csv")):
        print(f.name)
    raise FileNotFoundError("Could not find satellite signal pivot file.")

signal_pivot = pd.read_csv(signal_path)

signal_pivot["year"] = pd.to_numeric(
    signal_pivot["year"],
    errors="coerce"
).astype(int)

print("Loaded signal pivot:", signal_path.name)
print("Columns:", signal_pivot.columns.tolist())

display(signal_pivot.round(4))

## 4. July/August satellite bracket vs November WASDE

The July/August bracket tests whether the later November WASDE soybean yield falls between the July-end and August-end satellite-implied yield estimates. This is a benchmark test of whether the satellite signal captures the eventual WASDE yield range.

In [ ]:
# ============================================================
# July/August satellite bracket vs November WASDE yield
# ============================================================

bracket_test = signal_pivot.merge(
    nov_wasde[
        [
            "year",
            "nov_release_date",
            "nov_wasde_yield"
        ]
    ],
    on="year",
    how="inner",
    validate="one_to_one"
)

bracket_test["sat_bracket_lower"] = bracket_test[
    ["july_satellite_yield_adj", "aug_satellite_yield_adj"]
].min(axis=1)

bracket_test["sat_bracket_upper"] = bracket_test[
    ["july_satellite_yield_adj", "aug_satellite_yield_adj"]
].max(axis=1)

bracket_test["nov_inside_july_aug_bracket"] = (
    (bracket_test["nov_wasde_yield"] >= bracket_test["sat_bracket_lower"]) &
    (bracket_test["nov_wasde_yield"] <= bracket_test["sat_bracket_upper"])
)

bracket_test["distance_to_bracket"] = 0.0

bracket_test.loc[
    bracket_test["nov_wasde_yield"] < bracket_test["sat_bracket_lower"],
    "distance_to_bracket"
] = (
    bracket_test["sat_bracket_lower"] -
    bracket_test["nov_wasde_yield"]
)

bracket_test.loc[
    bracket_test["nov_wasde_yield"] > bracket_test["sat_bracket_upper"],
    "distance_to_bracket"
] = (
    bracket_test["nov_wasde_yield"] -
    bracket_test["sat_bracket_upper"]
)

bracket_display_cols = [
    "year",
    "july_satellite_yield_adj",
    "aug_satellite_yield_adj",
    "sat_bracket_lower",
    "sat_bracket_upper",
    "nov_wasde_yield",
    "nov_inside_july_aug_bracket",
    "distance_to_bracket"
]

display(bracket_test[bracket_display_cols].round(4))

In [ ]:
# ============================================================
# Bracket summary and tolerance checks
# ============================================================

bracket_summary = pd.DataFrame([{
    "num_years": len(bracket_test),
    "hits": int(bracket_test["nov_inside_july_aug_bracket"].sum()),
    "hit_rate": bracket_test["nov_inside_july_aug_bracket"].mean(),
    "mean_distance_to_bracket": bracket_test["distance_to_bracket"].mean(),
    "mean_distance_misses_only": bracket_test.loc[
        ~bracket_test["nov_inside_july_aug_bracket"],
        "distance_to_bracket"
    ].mean()
}])

display(bracket_summary.round(4))

summary_tol = []

for tol in [0.0, 0.5, 1.0]:
    col = f"inside_bracket_tol_{tol}"

    bracket_test[col] = (
        (bracket_test["nov_wasde_yield"] >= bracket_test["sat_bracket_lower"] - tol) &
        (bracket_test["nov_wasde_yield"] <= bracket_test["sat_bracket_upper"] + tol)
    )

    summary_tol.append({
        "tolerance_bu_acre": tol,
        "num_years": len(bracket_test),
        "hits": int(bracket_test[col].sum()),
        "hit_rate": bracket_test[col].mean(),
        "mean_distance_to_bracket": bracket_test["distance_to_bracket"].mean()
    })

summary_tol = pd.DataFrame(summary_tol)

display(summary_tol.round(4))

## 5. August satellite vs August WASDE trading signal

The trading signal compares the August-end satellite-implied yield against the August WASDE yield. If the satellite estimate is meaningfully below WASDE, the signal is bullish for soybean futures. If the satellite estimate is meaningfully above WASDE, the signal is bearish. A 0.5 bu/acre threshold is used to avoid trading on small yield gaps.

In [ ]:
# ============================================================
# Build August satellite vs August WASDE signal table
# ============================================================

aug_sat = signal_pivot[
    [
        "year",
        "aug_satellite_yield_adj"
    ]
].copy()

aug_sat = aug_sat.rename(columns={
    "aug_satellite_yield_adj": "satellite_yield_adj"
})

aug_wasde_std = aug_wasde.copy()

rename_map = {}
if "aug_release_date" in aug_wasde_std.columns:
    rename_map["aug_release_date"] = "release_date"
if "aug_release_time" in aug_wasde_std.columns:
    rename_map["aug_release_time"] = "release_time"

aug_wasde_std = aug_wasde_std.rename(columns=rename_map)

needed_cols = [
    "year",
    "release_date",
    "aug_wasde_yield",
    "aug_wasde_harvested_area",
    "aug_wasde_production",
    "aug_wasde_ending_stocks"
]

if "release_time" in aug_wasde_std.columns:
    needed_cols.insert(2, "release_time")

missing_cols = [c for c in needed_cols if c not in aug_wasde_std.columns]
if missing_cols:
    raise ValueError(f"Missing columns in aug_wasde_std: {missing_cols}")

aug_wasde_std = aug_wasde_std[needed_cols].copy()
aug_wasde_std["year"] = pd.to_numeric(aug_wasde_std["year"], errors="coerce").astype(int)

aug_trade_signal = aug_sat.merge(
    aug_wasde_std,
    on="year",
    how="inner",
    validate="one_to_one"
)

aug_trade_signal["yield_gap"] = (
    aug_trade_signal["satellite_yield_adj"] -
    aug_trade_signal["aug_wasde_yield"]
)

# WASDE units: bu/acre * million acres = million bushels
aug_trade_signal["satellite_production_adj"] = (
    aug_trade_signal["satellite_yield_adj"] *
    aug_trade_signal["aug_wasde_harvested_area"]
)

aug_trade_signal["production_gap"] = (
    aug_trade_signal["satellite_production_adj"] -
    aug_trade_signal["aug_wasde_production"]
)

# production_gap < 0 -> satellite says less supply than WASDE -> bullish
# production_gap > 0 -> satellite says more supply than WASDE -> bearish
aug_trade_signal["price_signal"] = -aug_trade_signal["production_gap"]

display_cols = [
    "year",
    "release_date",
    "satellite_yield_adj",
    "aug_wasde_yield",
    "yield_gap",
    "aug_wasde_harvested_area",
    "satellite_production_adj",
    "aug_wasde_production",
    "production_gap",
    "price_signal"
]

if "release_time" in aug_trade_signal.columns:
    display_cols.insert(2, "release_time")

display(aug_trade_signal[display_cols].round(4))

In [ ]:
# ============================================================
# Define long / short / no-trade position
# ============================================================

threshold = 0.5  # bu/acre

aug_trade_signal["position"] = 0

# satellite yield lower than WASDE
# => less supply than WASDE
# => bullish soybean price
# => long
aug_trade_signal.loc[
    aug_trade_signal["yield_gap"] <= -threshold,
    "position"
] = 1

# satellite yield higher than WASDE
# => more supply than WASDE
# => bearish soybean price
# => short
aug_trade_signal.loc[
    aug_trade_signal["yield_gap"] >= threshold,
    "position"
] = -1

aug_trade_signal["trade_direction"] = aug_trade_signal["position"].map({
    1: "long",
    -1: "short",
    0: "no_trade"
})

aug_trade_signal["price_signal_label"] = aug_trade_signal["position"].map({
    1: "bullish",
    -1: "bearish",
    0: "neutral"
})

display(
    aug_trade_signal[
        [
            "year",
            "release_date",
            "satellite_yield_adj",
            "aug_wasde_yield",
            "yield_gap",
            "production_gap",
            "price_signal",
            "position",
            "trade_direction",
            "price_signal_label"
        ]
    ].round(4)
)

## 6. Futures price response

The main futures response test uses Yahoo Finance continuous soybean futures (`ZS=F`) as a proxy. Returns are computed from the first available trading day on or after September 1 to 3/5/10/20 trading-day horizons and to the last trading day in October. Close-to-close returns are used as the main specification, with an open-close midpoint as a robustness proxy.

In [ ]:
# ============================================================
# Pull continuous soybean futures proxy from Yahoo Finance
# Ticker: ZS=F
# ============================================================

!pip install yfinance -q

import yfinance as yf

ticker = "ZS=F"

data = yf.download(
    ticker,
    start="2019-08-15",
    end="2025-11-05",
    progress=False,
    auto_adjust=False
)

data = data.reset_index()

if isinstance(data.columns, pd.MultiIndex):
    data.columns = [
        "_".join([str(x) for x in col if x])
        for col in data.columns
    ]

def find_col(base_name, ticker):
    candidates = [
        base_name,
        f"{base_name}_{ticker}",
    ]
    for c in candidates:
        if c in data.columns:
            return c
    return None

open_col = find_col("Open", ticker)
close_col = find_col("Close", ticker)

if open_col is None or close_col is None:
    print("Columns:", data.columns.tolist())
    raise ValueError("Open or Close column not found.")

soybean_continuous = data[["Date", open_col, close_col]].copy()

soybean_continuous = soybean_continuous.rename(columns={
    "Date": "date",
    open_col: "open",
    close_col: "close"
})

soybean_continuous["date"] = pd.to_datetime(soybean_continuous["date"])
soybean_continuous["open"] = pd.to_numeric(soybean_continuous["open"], errors="coerce")
soybean_continuous["close"] = pd.to_numeric(soybean_continuous["close"], errors="coerce")

soybean_continuous = soybean_continuous.dropna(subset=["open", "close"]).copy()

soybean_continuous["mid_oc"] = (
    soybean_continuous["open"] + soybean_continuous["close"]
) / 2

display(soybean_continuous.head())
display(soybean_continuous.tail())

In [ ]:
# ============================================================
# Function: build futures return table
# ============================================================

def build_sep_returns_proxy(price_df, price_col="close", entry_month=9, entry_day=1):
    """
    For each year, use the first available trading day on or after Sep 1
    as the entry date, then compute 3d/5d/10d/20d and Oct-end long returns.
    """

    df = price_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["year"] = df["date"].dt.year
    df = df.sort_values("date").reset_index(drop=True)

    results = []

    for year in range(2019, 2026):

        year_df = (
            df[df["year"] == year]
            .sort_values("date")
            .reset_index(drop=True)
            .copy()
        )

        if year_df.empty:
            continue

        entry_target = pd.Timestamp(year=year, month=entry_month, day=entry_day)
        entry_candidates = year_df[year_df["date"] >= entry_target].copy()

        if entry_candidates.empty:
            continue

        entry_pos = entry_candidates.index[0]
        entry_row = year_df.loc[entry_pos]

        entry_date = entry_row["date"]
        entry_price = entry_row[price_col]

        out = {
            "year": year,
            "entry_date": entry_date,
            "entry_price": entry_price,
            "price_col": price_col
        }

        for h in [3, 5, 10, 20]:
            exit_pos = entry_pos + h

            if exit_pos < len(year_df):
                exit_row = year_df.loc[exit_pos]

                out[f"exit_date_{h}d"] = exit_row["date"]
                out[f"exit_price_{h}d"] = exit_row[price_col]
                out[f"long_return_{h}d"] = exit_row[price_col] / entry_price - 1
            else:
                out[f"exit_date_{h}d"] = pd.NaT
                out[f"exit_price_{h}d"] = np.nan
                out[f"long_return_{h}d"] = np.nan

        oct_df = year_df[year_df["date"].dt.month == 10].copy()

        if not oct_df.empty:
            oct_end_row = oct_df.iloc[-1]

            out["exit_date_oct_end"] = oct_end_row["date"]
            out["exit_price_oct_end"] = oct_end_row[price_col]
            out["long_return_oct_end"] = oct_end_row[price_col] / entry_price - 1
        else:
            out["exit_date_oct_end"] = pd.NaT
            out["exit_price_oct_end"] = np.nan
            out["long_return_oct_end"] = np.nan

        results.append(out)

    return pd.DataFrame(results)

In [ ]:
# ============================================================
# Compute futures returns using close and open-close midpoint
# ============================================================

sep1_returns_close = build_sep_returns_proxy(
    soybean_continuous,
    price_col="close"
)

sep1_returns_mid = build_sep_returns_proxy(
    soybean_continuous,
    price_col="mid_oc"
)

display(
    sep1_returns_close[
        [
            "year",
            "long_return_3d",
            "long_return_5d",
            "long_return_10d",
            "long_return_20d",
            "long_return_oct_end"
        ]
    ].round(4)
)

display(
    sep1_returns_mid[
        [
            "year",
            "long_return_3d",
            "long_return_5d",
            "long_return_10d",
            "long_return_20d",
            "long_return_oct_end"
        ]
    ].round(4)
)

## 7. Strategy returns and hit-rate summary

Strategy returns are computed as `position × long_return`, where `position = 1` is long, `position = -1` is short, and `position = 0` is no trade.

In [ ]:
# ============================================================
# Merge signal table with futures returns
# ============================================================

def build_strategy_returns(signal_df, returns_df):
    strategy = signal_df.merge(
        returns_df,
        on="year",
        how="inner",
        validate="one_to_one"
    )

    for h in [3, 5, 10, 20]:
        strategy[f"strategy_return_{h}d"] = (
            strategy["position"] * strategy[f"long_return_{h}d"]
        )
        strategy[f"hit_{h}d"] = strategy[f"strategy_return_{h}d"] > 0

    strategy["strategy_return_oct_end"] = (
        strategy["position"] * strategy["long_return_oct_end"]
    )
    strategy["hit_oct_end"] = strategy["strategy_return_oct_end"] > 0

    return strategy


strategy_close = build_strategy_returns(aug_trade_signal, sep1_returns_close)
strategy_mid = build_strategy_returns(aug_trade_signal, sep1_returns_mid)

display(
    strategy_close[
        [
            "year",
            "trade_direction",
            "position",
            "yield_gap",
            "long_return_3d",
            "strategy_return_3d",
            "long_return_5d",
            "strategy_return_5d",
            "long_return_10d",
            "strategy_return_10d",
            "long_return_20d",
            "strategy_return_20d",
            "long_return_oct_end",
            "strategy_return_oct_end"
        ]
    ].round(4)
)

In [ ]:
# ============================================================
# Strategy hit-rate summary by horizon
# ============================================================

def summarize_strategy_by_horizon(strategy_df):
    horizon_summary = []

    for h in [3, 5, 10, 20]:
        col = f"strategy_return_{h}d"

        valid = strategy_df[
            strategy_df["position"] != 0
        ].dropna(subset=[col])

        horizon_summary.append({
            "horizon": f"{h}d",
            "num_trades": len(valid),
            "hit_rate": (valid[col] > 0).mean(),
            "avg_strategy_return": valid[col].mean(),
            "median_strategy_return": valid[col].median(),
            "total_strategy_return": valid[col].sum()
        })

    valid = strategy_df[
        strategy_df["position"] != 0
    ].dropna(subset=["strategy_return_oct_end"])

    horizon_summary.append({
        "horizon": "oct_end",
        "num_trades": len(valid),
        "hit_rate": (valid["strategy_return_oct_end"] > 0).mean(),
        "avg_strategy_return": valid["strategy_return_oct_end"].mean(),
        "median_strategy_return": valid["strategy_return_oct_end"].median(),
        "total_strategy_return": valid["strategy_return_oct_end"].sum()
    })

    return pd.DataFrame(horizon_summary)


horizon_summary_close = summarize_strategy_by_horizon(strategy_close)
horizon_summary_mid = summarize_strategy_by_horizon(strategy_mid)

display(horizon_summary_close.round(4))
display(horizon_summary_mid.round(4))

## 8. Save output tables

The saved tables are used for GitHub documentation, README summaries, and later paper/report drafting.

In [ ]:
# ============================================================
# Save outputs
# ============================================================

aug_wasde.to_csv(
    OUTPUT_TABLE_DIR / "august_wasde_soybean_benchmark.csv",
    index=False
)

nov_wasde.to_csv(
    OUTPUT_TABLE_DIR / "november_wasde_soybean_benchmark.csv",
    index=False
)

bracket_test.to_csv(
    OUTPUT_TABLE_DIR / "july_aug_satellite_vs_november_wasde_bracket_test.csv",
    index=False
)

bracket_summary.to_csv(
    OUTPUT_TABLE_DIR / "july_aug_satellite_vs_november_wasde_bracket_summary.csv",
    index=False
)

summary_tol.to_csv(
    OUTPUT_TABLE_DIR / "july_aug_satellite_vs_november_wasde_bracket_tolerance_summary.csv",
    index=False
)

aug_trade_signal.to_csv(
    OUTPUT_TABLE_DIR / "august_satellite_vs_wasde_signal_table.csv",
    index=False
)

soybean_continuous.to_csv(
    OUTPUT_TABLE_DIR / "soybean_continuous_yahoo_zs_f.csv",
    index=False
)

sep1_returns_close.to_csv(
    OUTPUT_TABLE_DIR / "soybean_futures_returns_sep1_close.csv",
    index=False
)

sep1_returns_mid.to_csv(
    OUTPUT_TABLE_DIR / "soybean_futures_returns_sep1_mid_oc.csv",
    index=False
)

strategy_close.to_csv(
    OUTPUT_TABLE_DIR / "august_satellite_wasde_strategy_returns_close.csv",
    index=False
)

strategy_mid.to_csv(
    OUTPUT_TABLE_DIR / "august_satellite_wasde_strategy_returns_mid_oc.csv",
    index=False
)

horizon_summary_close.to_csv(
    OUTPUT_TABLE_DIR / "august_satellite_wasde_strategy_horizon_summary_close.csv",
    index=False
)

horizon_summary_mid.to_csv(
    OUTPUT_TABLE_DIR / "august_satellite_wasde_strategy_horizon_summary_mid_oc.csv",
    index=False
)

print("Saved output tables to:", OUTPUT_TABLE_DIR)

print("\nSaved files:")
for f in sorted(OUTPUT_TABLE_DIR.glob("*.csv")):
    if any(key in f.name for key in ["wasde", "bracket", "strategy", "futures_returns", "soybean_continuous"]):
        print(f.name)

## Interpretation notes

- The July/August satellite bracket is a yield benchmark test, not a trading strategy by itself.
- The August satellite-WASDE yield gap is used for the futures signal because both inputs are available around the August WASDE release period.
- The futures response test is exploratory. A 0.5 bu/acre threshold avoids trading on small satellite-WASDE gaps.
- Close-to-close returns are the main proxy. Open-close midpoint returns are included as a robustness check, not as an exact executable trading price.